# Test STS-Miner on a TSMC2014 NYC subset

This notebook loads 20,000 instances from the 20 most common event types in the preprocessed NYC check-in dataset, prepares local meter coordinates, and passes the DataFrame into `STSMiner`. The miner module itself does not load any dataset from disk.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "Notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "Source"))

from sts_miner import STSMiner, add_local_meter_coordinates, patterns_to_frame

In [ ]:
data_path = PROJECT_ROOT / "Data" / "Preprocessed" / "dataset_TSMC2014_NYC_preprocessed.csv"

raw_data = pd.read_csv(data_path)
common_event_types = raw_data["Event_type"].value_counts().head(20).index
events = raw_data[raw_data["Event_type"].isin(common_event_types)].head(1000).copy()
events = add_local_meter_coordinates(events)

events[["Event_ID", "Event_type", "timestamp", "longitude", "latitude", "x_meters", "y_meters"]].head()

,Event_ID,Event_type,timestamp,longitude,latitude,x_meters,y_meters
2,3,Home (private),2.250000,-73.883070,40.716162,32654.958989,16109.739692
3,4,Medical Center,2.533333,-73.982519,40.745164,24278.238688,19334.627759
5,6,Food & Drink Shop,3.850000,-73.954687,40.690427,26622.571392,13248.186430
6,7,Coffee Shop,4.483333,-73.974121,40.751591,24985.562583,20049.347755
7,8,Bus Station,4.550000,-73.955341,40.779422,26567.453823,23143.935847


In [3]:
events.shape, events["Event_type"].nunique(), events["Event_type"].value_counts()

((4000, 8),
 20,
 Event_type
 Home (private)                              515
 Office                                      389
 Bar                                         375
 Coffee Shop                                 268
 Subway                                      266
 Gym / Fitness Center                        245
 Food & Drink Shop                           221
 Train Station                               208
 Other Great Outdoors                        186
 Park                                        180
 Bus Station                                 148
 Neighborhood                                133
 College Academic Building                   133
 American Restaurant                         129
 Deli / Bodega                               123
 Road                                        119
 Building                                    111
 Residential Building (Apartment / Condo)    103
 Medical Center                               78
 Clothing Store                         

In [4]:
miner = STSMiner(
    events,
    spatial_radius=500.0,
    temporal_window=60.0,
    min_sequence_index=1.0,
    max_length=None,
)

patterns = miner.mine()
patterns_frame = patterns_to_frame(patterns)
patterns_frame

KeyboardInterrupt: 

In [ ]:
assert not patterns_frame.empty
assert patterns_frame["sequence_index"].min() >= 1.0

patterns_frame.head(10)

,sequence,length,sequence_index,tail_event_count,last_density_ratio
0,Park -> Clothing Store,2,749.198040,5,749.198040
1,Park -> Clothing Store -> Train Station,3,559.123723,2,559.123723
2,Park -> Clothing Store -> Coffee Shop,3,529.696158,1,529.696158
3,Park -> Clothing Store -> Coffee Shop -> Gym /...,4,529.696158,1,621.248581
4,Park -> Clothing Store -> Coffee Shop -> Gym /...,5,529.696158,1,882.826930
5,Park -> Clothing Store -> Coffee Shop -> Gym /...,6,529.696158,1,571.831080
6,Park -> Clothing Store -> Coffee Shop -> Gym /...,6,529.696158,1,1198.122263
7,Park -> Clothing Store -> Coffee Shop -> Gym /...,7,529.696158,1,571.831080
8,Park -> Clothing Store -> Coffee Shop -> Train...,4,529.696158,1,931.872871
9,Clothing Store -> Train Station,2,358.412643,4,358.412643
